# Selective Recurrence And A Mamba Toy

This notebook extends the fixed R07 recurrence from `20_state_space_models.ipynb` with a bounded input-dependent gate. The goal is narrow: isolate when selective memory helps on a tiny conditional-retention task, and show a clear failure mode when the control signal is too weak.

## Motivation

Structured state-space models give a linear recurrent backbone that can be implemented efficiently [@structured_state_spaces_2021]. Mamba is a selective state-space family: some sequence dynamics become input-dependent instead of staying fixed at every timestep [@mamba_2023]. This notebook does **not** implement Mamba. It builds one scalar toy that isolates the selective-update idea.

That isolation matters because related post-attention families make different tradeoffs. Hyena is a long-convolution family rather than a gated scalar recurrence [@hyena_hierarchy_2023]. RWKV is a gated recurrent family with its own training-time and inference-time story [@rwkv_2023]. So the right claim here is small: selective recurrence can help on a task that requires conditional overwrite and conditional retention.

## Hypothesis

If a recurrence can use an input-dependent gate, then it should solve a conditional-retention toy task that no single fixed gate solves well. If the control signal is weak or ambiguous, the benefit should collapse because the gate stops behaving like a near-binary decision.

## Baseline

The baseline is the fixed scalar recurrence from Task 10, restricted to the same update family that appears when the selective gate is constant:

`x_t = alpha x_{t-1} + beta u_t`, with `alpha = g a` and `beta = (1 - g) b`.

We sweep constant `g` values and compare the best fixed recurrence against the selective toy on the same sequence.

## Metric

The target is the ideal memory trace: after a write step, the state should hold the most recent payload until the next write. We report mean-squared error (MSE) against that target trace. We also check two mechanistic properties: the sigmoid gates stay in `[0, 1]`, and constant gates reduce exactly to the fixed recurrence.

## Mathematical derivation

We study the scalar selective recurrence

`x_t = g_t a x_{t-1} + (1 - g_t) b u_t`, `y_t = c x_t`, with `g_t in [0, 1]`.

Interpretation:

- `g_t` near `1`: retain the previous state.
- `g_t` near `0`: overwrite with the new input.

If the gate is constant, `g_t = g`, then the recurrence becomes

`x_t = (g a) x_{t-1} + ((1 - g) b) u_t`,

which is exactly the fixed linear recurrence from Task 10 with effective transition `g a` and effective input gain `(1 - g) b`. So selectivity does not add a new algebraic family here; it adds timestep-dependent control over the existing recurrence.

For the conditional-retention toy, the useful regime is simple: write timesteps should produce `g_t approx 0`, while distractor timesteps should produce `g_t approx 1`. A sigmoid controller `g_t = sigma(w z_t + b)` gives us a bounded soft version of that rule.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.state_space import (
    selective_scalar_linear_recurrence,
    selective_sigmoid_gates,
    scalar_linear_recurrence,
)

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## PyTorch implementation

The toy sequence has two channels collapsed into two 1D arrays:

- `payloads[t]`: the value we want to store on write steps
- `write_mask[t]`: whether this timestep should overwrite memory

We convert `write_mask` into a control signal `z_t` where write steps map to `-1` and retain steps map to `+1`. A strong positive sigmoid weight then makes the gate nearly binary without making it exactly discrete.

In [ ]:
def conditional_retention_target(payloads: torch.Tensor, write_mask: torch.Tensor) -> torch.Tensor:
    state = torch.tensor(0.0, dtype=payloads.dtype, device=payloads.device)
    targets = []
    for payload_t, write_t in zip(payloads, write_mask):
        if write_t > 0:
            state = payload_t
        targets.append(state.clone())
    return torch.stack(targets)

def run_selective_case(
    payloads: torch.Tensor,
    control_signal: torch.Tensor,
    *,
    gate_weight: float,
    gate_bias: float = 0.0,
    transition: float = 1.0,
    input_gain: float = 1.0,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    gates = selective_sigmoid_gates(
        control_signal,
        weight=torch.tensor(gate_weight, dtype=payloads.dtype),
        bias=torch.tensor(gate_bias, dtype=payloads.dtype),
    )
    states, outputs = selective_scalar_linear_recurrence(
        payloads,
        gates=gates,
        transition=torch.tensor(transition, dtype=payloads.dtype),
        input_gain=torch.tensor(input_gain, dtype=payloads.dtype),
        output_gain=torch.tensor(1.0, dtype=payloads.dtype),
    )
    return gates, states, outputs

def run_fixed_case(
    payloads: torch.Tensor,
    *,
    gate_value: float,
    transition: float = 1.0,
    input_gain: float = 1.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    effective_transition = torch.tensor(gate_value * transition, dtype=payloads.dtype)
    effective_input_gain = torch.tensor((1.0 - gate_value) * input_gain, dtype=payloads.dtype)
    return scalar_linear_recurrence(
        payloads,
        transition=effective_transition,
        input_gain=effective_input_gain,
        output_gain=torch.tensor(1.0, dtype=payloads.dtype),
    )

payloads = torch.tensor([2.0, 0.0, 0.0, -1.5, 0.0, 0.0, 3.0, 0.0], dtype=torch.float64)
write_mask = torch.tensor([1, 0, 0, 1, 0, 0, 1, 0], dtype=torch.float64)
control_signal = torch.where(
    write_mask > 0,
    torch.tensor(-1.0, dtype=torch.float64),
    torch.tensor(1.0, dtype=torch.float64),
)
target_trace = conditional_retention_target(payloads, write_mask)

success_gates, success_states, success_outputs = run_selective_case(
    payloads,
    control_signal,
    gate_weight=8.0,
)
success_mse = torch.mean((success_outputs - target_trace) ** 2).item()

fixed_grid = torch.linspace(0.0, 1.0, 101, dtype=torch.float64)
fixed_runs = []
for gate_value in fixed_grid.tolist():
    _, fixed_outputs = run_fixed_case(payloads, gate_value=gate_value)
    mse = torch.mean((fixed_outputs - target_trace) ** 2).item()
    fixed_runs.append((gate_value, mse, fixed_outputs))
best_fixed_gate, best_fixed_mse, best_fixed_outputs = min(fixed_runs, key=lambda item: item[1])

weak_gates, weak_states, weak_outputs = run_selective_case(
    payloads,
    control_signal,
    gate_weight=0.5,
)
weak_mse = torch.mean((weak_outputs - target_trace) ** 2).item()

inverted_gates, inverted_states, inverted_outputs = run_selective_case(
    payloads,
    -control_signal,
    gate_weight=8.0,
)
inverted_mse = torch.mean((inverted_outputs - target_trace) ** 2).item()

print('target trace        :', target_trace)
print('selective outputs   :', success_outputs)
print('best fixed outputs  :', best_fixed_outputs)
print('weak-gate outputs   :', weak_outputs)
print()
print(f'selective success MSE: {success_mse:.8f}')
print(f'best fixed gate      : {best_fixed_gate:.3f} (MSE={best_fixed_mse:.6f})')
print(f'weak selective MSE   : {weak_mse:.6f}')
print(f'inverted-control MSE : {inverted_mse:.6f}')
print()
print('success gates:', success_gates)
print('weak gates   :', weak_gates)

## Numerical checks

These checks encode the acceptance criteria directly:

1. Sigmoid gates are bounded.
2. Constant gates reduce to the fixed recurrence.
3. A strong control signal solves the conditional-retention task much better than any fixed gate.
4. Weak or incorrect control signals produce visible failure cases.

In [ ]:
assert torch.all(success_gates >= 0.0)
assert torch.all(success_gates <= 1.0)
assert torch.all(weak_gates >= 0.0)
assert torch.all(weak_gates <= 1.0)

constant_gates = torch.full_like(payloads, 0.7)
selective_constant_states, selective_constant_outputs = selective_scalar_linear_recurrence(
    payloads,
    gates=constant_gates,
    transition=torch.tensor(0.9, dtype=torch.float64),
    input_gain=torch.tensor(1.3, dtype=torch.float64),
    output_gain=torch.tensor(1.1, dtype=torch.float64),
)
fixed_states, fixed_outputs = scalar_linear_recurrence(
    payloads,
    transition=torch.tensor(0.7 * 0.9, dtype=torch.float64),
    input_gain=torch.tensor((1.0 - 0.7) * 1.3, dtype=torch.float64),
    output_gain=torch.tensor(1.1, dtype=torch.float64),
)
torch.testing.assert_close(selective_constant_states, fixed_states, atol=1e-12, rtol=1e-12)
torch.testing.assert_close(selective_constant_outputs, fixed_outputs, atol=1e-12, rtol=1e-12)

assert success_mse < 1e-5
assert best_fixed_mse > 2.0
assert weak_mse > 1.0
assert inverted_mse > weak_mse

print('All numerical checks passed.')

## Ablations

The ablations separate three regimes:

- **Strong selective gate**: nearly binary overwrite/retain decisions.
- **Weak selective gate**: gates remain too soft, so memory updates blur together.
- **Incorrect control**: the mechanism is expressive enough, but the controller tells it to do the wrong thing.

This is the practical lesson: selectivity only helps when the gating signal is informative.

In [ ]:
ablation_rows = [
    ('strong selective', success_mse, success_gates.min().item(), success_gates.max().item()),
    ('weak selective', weak_mse, weak_gates.min().item(), weak_gates.max().item()),
    ('inverted control', inverted_mse, inverted_gates.min().item(), inverted_gates.max().item()),
]

for name, mse, gate_min, gate_max in ablation_rows:
    print(f'{name:>16} | mse={mse:>10.6f} | gate range=({gate_min:>7.4f}, {gate_max:>7.4f})')

gap = best_fixed_mse - success_mse
print()
print(f'best fixed minus selective MSE gap: {gap:.6f}')

## Assumptions

- The task exposes the mechanism cleanly: write steps are known and represented by a one-dimensional control feature.
- The recurrence is scalar, so there is no hidden-state mixing across channels.
- The controller is hand-designed, not learned. We are probing representational behavior, not optimization.

## Risks

- The toy can overstate confidence because the gating signal is cleaner than real sequence data.
- Good performance here does not tell us about throughput, training stability, or language-model quality.
- Mamba, Hyena, and RWKV belong to different architecture families, so this toy should not be used to rank them globally.

## Failure criteria

This notebook fails its purpose if any of the following happens:

- the gates leave `[0, 1]`
- constant gates do not reduce to the fixed recurrence
- the selective model does not beat the best fixed gate on the conditional-retention task
- the notebook hides failure cases instead of showing at least one weak-control or wrong-control breakdown

## Exercises

See the paired exercise sheet at `exercises/research/21_selective_recurrence_mamba_toy.md` and the worked solutions at `exercises/research/solutions/21_selective_recurrence_mamba_toy_solutions.md`.

Suggested follow-up prompts:

1. Re-derive the constant-gate reduction and verify the effective parameters.
2. Design a two-channel task where one channel controls the gate and the other provides distractor content.
3. Explain why a weak gate behaves like a lossy moving average instead of a crisp memory controller.

## References

- Gu et al., *Efficiently Modeling Long Sequences with Structured State Spaces* [@structured_state_spaces_2021]
- Gu and Dao, *Mamba: Linear-Time Sequence Modeling with Selective State Spaces* [@mamba_2023]
- Poli et al., *Hyena Hierarchy: Towards Larger Convolutional Language Models* [@hyena_hierarchy_2023]
- Peng et al., *RWKV: Reinventing RNNs for the Transformer Era* [@rwkv_2023]